# 單元 14.9 APCS 實作真題特訓（初級題）：g595. 修補圍籬

**適合對象**：程式設計初學者（完全零基礎） / APCS 扎根學習者  
**對應教材**：吳邦一老師《Python 程式設計從 APCS 實作 1 級到 3 級》第 11 章（第 42–43 頁）  
**題型定位**：APCS 實作真題特訓——初級題（對應舊版實作第 1 題，APCS 2021-11 場次）  

---

### 🗺️ 本單元學習地圖與通關導航
在前面的單元中，我們征服了三邊長判斷、大數差值、邏輯運算子、比分統計、購物車抵銷、人力分配二次函數、購買力全距以及七言對聯格律。
現在，我們終於來到了**第十四章 APCS 實作真題特訓（初級題）的最終壓軸大關——g595. 修補圍籬**！

這道題目模擬農場生活中的實用修繕問題：颱風過後，農場一整排圍籬中部分木樁損壞（高度為 0）。為了節省修繕材料並兼顧防護效果，農場主人決定**將每一處損壞的木樁，修補到其「左右相鄰兩根木樁中較矮的那一根」的高度**。

這道題目表面上看似只是簡單的 `min()` 取較小值，但它暗藏了所有初學程式設計者在陣列操作時最容易崩潰的**「兩端邊界存取陷阱（Boundary Edge Cases）」**！如果直接寫 `h[i-1]` 或 `h[i+1]`，在最左端會發生索引負數倒轉，在最右端更會直接觸發 `IndexError: list index out of range` 當場爆掉！

我們將這道經典大題拆解為 6 個循序漸進的學習階梯：
1. **14.9.1 題意解析：損壞高度 0 圍籬修復成本最小化**：以修補農場木樁的生動比喻，理解取左右鄰居較小高度的成本最小化邏輯。
2. **14.9.2 一維串列相鄰探測與非零高度提取**：掌握串列走訪掃描技巧，定位所有高度為 0 的損壞位置。
3. **14.9.3 邊界特例分析：最左端第 0 格與最右端最後一格**：深入剖析兩端無鄰居的極限情況，建立第一道條件分支防禦線。
4. **14.9.4 中間損壞格取相鄰兩側較小值：`min(left, right)`**：結合 `min()` 函數，完成內部一般格與邊界格的完整分流判定。
5. **14.9.5 總花費累加器與完整 AC 程式碼實作**：串聯計數累加器，手把手組裝 12 行考場滿分代碼。
6. **14.9.6 加框技巧延伸：前後各補虛擬圍籬免邊界特判**：解鎖競賽選手必學的「哨兵加框神技（Padding Sentinel）」，將程式碼極限壓縮至 7 行！

帶上你的工具箱，讓我們一起修好這排圍籬，為第十四章初級真題特訓畫上最完美的榮耀句點！

## 📜 【APCS 官方完整真題題面與規範】g595. 修補圍籬

> 📌 **題目資訊快覽**  
> * **題目名稱**：修補圍籬 (Repair Fence)  
> * **題目出處**：APCS 大學程式設計先修檢測（2021 年 11 月場次）實作題第 1 題（初級題）  
> * **線上評判**：ZeroJudge g595 / 高中生程式解題系統  
> * **對應講義**：吳邦一老師《Python 程式設計從 APCS 實作 1 級到 3 級》第 11 章（第 42–43 頁）  
> * **難度評級**：★☆☆☆☆（初級題 / 一維陣列相鄰探測、邊界條件防禦、極值運算、哨兵加框）  

---

### 📝 題目描述（Problem Description）
農場裡有一排由 $n$ 根木樁組成的圍籬，由左至右依序編號為 $0$ 到 $n - 1$。
每根木樁的高度記錄在一個整數序列 $h_0, h_1, \dots, h_{n-1}$ 中。
經歷了一場強烈颱風後，有些木樁被吹斷了，其高度變成了 **`0`**。

農場主人決定修補這些損壞的木樁，修復規則如下：
1. **中間損壞木樁**：若位置 $i$（$0 < i < n - 1$）的木樁高度為 0，其修補後的**高度為其「左邊鄰居」與「右邊鄰居」中較小的那一個高度**，即：  
   $$\text{Cost} = \min(h_{i-1}, h_{i+1})$$
2. **最左端損壞木樁**：若最左端位置 $0$ 的木樁高度為 0，因為它沒有左邊鄰居，所以其修補後的**高度直接等於其右邊鄰居的高度**，即：  
   $$\text{Cost} = h_1$$
3. **最右端損壞木樁**：若最右端位置 $n - 1$ 的木樁高度為 0，因為它沒有右邊鄰居，所以其修補後的**高度直接等於其左邊鄰居的高度**，即：  
   $$\text{Cost} = h_{n-2}$$

⚠️ **重要題目保證**：
題目保證**「不會有連續兩根相鄰的木樁同時損壞」**（即序列中絕對不會出現連續兩個 0）。這意味著任何損壞木樁的相鄰鄰居，其高度必定大於 0！

修補每根損壞木樁的費用等於其修補後的高度。請計算並輸出**修補所有損壞木樁所需的總費用**。

---

### 📥 輸入說明（Input Format）
* **第 1 行**：包含一個正整數 $n$（$2 \le n \le 100$），代表圍籬木樁的總數量。
* **第 2 行**：包含 $n$ 個非負整數 $h_0, h_1, \dots, h_{n-1}$（$0 \le h_i \le 100$），代表各木樁的高度，數值間以半形空格隔開。

---

### 📤 輸出說明（Output Format）
* 輸出僅有一行，包含一個非負整數，代表**修補所有木樁的總花費**。
* 若圍籬完全沒有損壞（無任何 0），請輸出 `0`。
* 結尾請換行，末端請勿輸出多餘的空白字元。

---

### 📋 官方範例與典型測資一覽表（Sample Cases）

| 範例編號 | 輸入範例（Input） | 輸出範例（Output） | 各損壞木樁位置與修補高度計算詳細說明 |
| :---: | :--- | :--- | :--- |
| **範例 1**<br>（官方範例 1） | `3`<br>`2 0 4` | `2` | 圍籬長度 $n=3$，高度為 `[2, 0, 4]`。<br>唯一的損壞位於中間索引 1（$h_1=0$）。<br>左邊鄰居為 $h_0=2$，右邊鄰居為 $h_2=4$。<br>修補費用為 $\min(2, 4) = 2$。<br>總花費為 `2`。 |
| **範例 2**<br>（官方範例 2） | `9`<br>`0 5 3 0 6 4 0 1 0` | `10` | 圍籬長度 $n=9$。共有 4 處損壞：<br>1. 索引 0（最左端）：右鄰居為 5，修補花費 **5**；<br>2. 索引 3（中間）：左鄰 3、右鄰 6，修補花費 $\min(3, 6) =$ **3**；<br>3. 索引 6（中間）：左鄰 4、右鄰 1，修補花費 $\min(4, 1) =$ **1**；<br>4. 索引 8（最右端）：左鄰居為 1，修補花費 **1**。<br>總花費 $= 5 + 3 + 1 + 1 =$ `10`。 |
| **範例 3**<br>（全無損壞邊界） | `4`<br>`5 6 7 8` | `0` | 圍籬完全沒有任何損壞（無 0），總花費為 `0`。 |
| **範例 4**<br>（兩端極小邊界） | `2`<br>`0 8` | `8` | 圍籬長度 $n=2$。索引 0 損壞，右鄰居為 8，修補花費為 `8`。 |

---

### ⚙️ 評分說明與測資範圍限制
* 木樁總數：$2 \le n \le 100$。
* 木樁高度：$0 \le h_i \le 100$。
* 題目保證：序列中不會出現連續兩個 0。
* 執行時間限制：1.0 秒（$n \le 100$ 線性掃描僅需數微秒，輕量通過）。
* 記憶體限制：64 MB。
* 評分機制：本題各測試點獨立計分，輸出完全符合規格即獲得 100 分滿分。

### 14.9.1 題意解析：損壞高度 0 圍籬修復成本最小化

#### 🌾 生活化比喻：農場主人的實用美學
想像你接管了一座綠草如茵的牧場，牧場周圍立著一整排木樁圍籬，用來防止羊群走失。
經歷了一場暴風雨後，有些木樁不幸斷裂倒塌（在資料中以高度 `0` 表示）。身為農場主人，你必須採購新的木材將這些空缺補齊。
你可能會問：「該補多高呢？」
1. 如果補得太高（比如比兩邊都高），不僅浪費昂貴的木料，視覺上還會突出一大塊，非常突兀；
2. 如果補得太矮，羊群可能會從凹槽處跳出去；
3. 最經濟且美觀的修補標準，就是**「看兩側鄰居的高度，取兩者中較矮的那一根的高度」**（$\min(\text{左鄰}, \text{右鄰})$）！這樣既能節省預算，又能讓圍籬平順地銜接在一起。

#### 🛡️ 題目最暖心的護城河：不會出現連續兩個 0
在演算法問題中，這種相鄰取值的題目最怕遇到「骨牌連鎖斷裂」（例如連續三個木樁都是 0，大家都想參考鄰居，結果鄰居也是 0）。
而本題給予了我們一條無比珍貴的保證：
> **題目保證序列中絕對不會有相鄰的木樁同時斷裂！**

這意味著：只要你在位置 $i$ 發現了 `h[i] == 0`，你可以百分之百確信：**它的鄰居（如果有存在的話）其高度絕對大於 0**！我們完全不需要擔心鄰居也是 0 的死結連鎖問題，這極大地簡化了程式的思考維度。

#### 🧩 演算法核心戰略
我們的任務非常明確：
從左到右走訪圍籬陣列，每當看見 `h[i] == 0`，就根據其位置計算填補費用，並加總到累加器中。當全部巡視完畢後，輸出總金額即可！

In [ ]:
# ==============================================================================
# 14.9.1 [2] Code 範例區：圍籬木樁資料與損壞位置識別展示
# ==============================================================================

# 官方範例 1 的圍籬資料
n = 3
h = [2, 0, 4]

print(f"圍籬木樁總數：{n}")
print(f"各木樁初始高度：{h}")
print("-" * 50)

# 巡邏檢查每一根木樁
for i in range(n):
    if h[i] == 0:
        left_neighbor = h[i - 1]
        right_neighbor = h[i + 1]
        repair_height = min(left_neighbor, right_neighbor)
        print(f"發現損壞木樁位於索引 {i}！")
        print(f"  左邊鄰居高度：{left_neighbor}")
        print(f"  右邊鄰居高度：{right_neighbor}")
        print(f"  修補費用 min({left_neighbor}, {right_neighbor}) = {repair_height}")

print("-" * 50)


In [ ]:
# ==============================================================================
# 14.9.1 [3] Code 填空題
# 任務說明：給定一個圍籬串列 h = [5, 0, 8]，請補齊下方代碼中的挖空處 ___，
#           判斷損壞位置並取出左右兩側較小的高度。
# ==============================================================================

h = [5, 0, 8]
broken_idx = 1

# 提示：判斷當前木樁是否為 0
if h[broken_idx] == ___:
    # 提示：取出左鄰 h[0] 與右鄰 h[2]，並使用 min() 求較小者
    cost = min(h[___], h[___])
    print(f"修復花費為：{cost}")

# 預期輸出：修復花費為：5


In [ ]:
# ==============================================================================
# 14.9.1 [4] Code 練習題
# 任務說明：給定一個圍籬串列 h（長度為 3 且保證只有中間第 1 格損壞），
#           請計算修復中間損壞木樁的花費並印出：修復費用 = {cost}。
#
# 【公開測試資料 1】
# 輸入 / 設定：h = [2, 0, 4]
# 預期輸出：修復費用 = 2
#
# 【公開測試資料 2】
# 輸入 / 設定：h = [10, 0, 6]
# 預期輸出：修復費用 = 6
# ==============================================================================
# 請在下方撰寫你的程式碼並執行測試：

h = [2, 0, 4]  # 可自行替換為測試資料 2 進行驗證



In [ ]:
# ==============================================================================
# 14.9.1 [5] Code 挑戰題
# 任務說明：假設農場主人改採「最高防禦原則」，規定修補高度必須取「左右鄰居中較大者」！
#           給定 h = [3, 0, 7]，請撰寫程式碼計算在此新規則下的修復費用並印出！
# （本題為自由挑戰題，無公開測資，請依題目情境自行思考並撰寫完整程式碼）
# ==============================================================================
# 請在下方撰寫你的程式碼：



### 14.9.2 一維串列相鄰探測與非零高度提取

#### 🚶 巡視走訪：線性掃描所有木樁
在處理整排木樁時，最基本的步驟就是透過一個循序走訪的迴圈，逐一檢查每一根木樁的高度：
```python
for i in range(n):
    if h[i] == 0:
        # 鎖定損壞木樁位置 i
```
當我們鎖定了某個損壞位置 $i$ 時，我們要探測它的**相鄰元素（Adjacent Elements）**：
- **左邊鄰居**：位於位置 $i - 1$，其高度為 `h[i - 1]`；
- **右邊鄰居**：位於位置 $i + 1$，其高度為 `h[i + 1]`。

#### ⚠️ 潛伏的索引越界危機（The Out-of-Bounds Danger）
初學同學在第一次寫這道題時，最常犯的直覺錯誤就是毫不猶豫地直接寫：
```python
cost = min(h[i - 1], h[i + 1])  # 🚨 未經保護的危險操作！
```
如果損壞木樁剛好位於中間（例如 $i = 3$），那麼 $i - 1 = 2$，$i + 1 = 4$，這樣存取完全合法。
但請試想以下兩種極限情況：
1. **如果損壞的是第 0 格（$i = 0$）**：
   $i - 1 = -1$！在 Python 中，`h[-1]` 不會報錯，但它會**繞到串列的最後一個元素**！這根本不是最左端木樁的真實左鄰居，會導致計算出莫名其妙的荒謬答案！
2. **如果損壞的是最後一格（$i = n - 1$）**：
   $i + 1 = n$！此時執行 `h[i + 1]` 會立刻引發 **`IndexError: list index out of range`**，程式當場崩潰暴斃！

因此，我們在提取鄰居高度之前，必須具備強烈的「邊界意識」，先辨識當前位置是否為極端邊界！

In [ ]:
# ==============================================================================
# 14.9.2 [2] Code 範例區：走訪串列並安全提取中間木樁的相鄰高度
# ==============================================================================

fence = [4, 5, 0, 8, 3]
n = len(fence)

print(f"圍籬序列：{fence}")
print("-" * 50)

for i in range(n):
    if fence[i] == 0:
        print(f"發現 0 位於索引 i = {i}")
        # 檢驗是否在中間安全區間（0 < i < n - 1）
        if 0 < i < n - 1:
            left_h = fence[i - 1]
            right_h = fence[i + 1]
            print(f"  處於安全中間區間！左鄰={left_h}, 右鄰={right_h}")
            print(f"  計算所需高度：min({left_h}, {right_h}) = {min(left_h, right_h)}")

print("-" * 50)


In [ ]:
# ==============================================================================
# 14.9.2 [3] Code 填空題
# 任務說明：請補齊下方代碼中的挖空處 ___，完成一個判斷索引 i 是否為中間安全位置
#           （即左右鄰居皆合法存在）的布林運算式。
# ==============================================================================

n = 5

def is_middle_index(i, length):
    # 提示：i 必須嚴格大於 0，且嚴格小於 length - 1
    return (i > ___) and (i < ___)

print(f"索引 0 是否為中間：{is_middle_index(0, n)}")
print(f"索引 2 是否為中間：{is_middle_index(2, n)}")
print(f"索引 4 是否為中間：{is_middle_index(4, n)}")
# 預期輸出：False, True, False


In [ ]:
# ==============================================================================
# 14.9.2 [4] Code 練習題
# 任務說明：給定一個圍籬串列 h，已知內部某個位置為 0（且保證不是在頭尾兩端）。
#           請寫一個迴圈找出該 0 的位置，並印出其左右鄰居的值：左鄰={left}, 右鄰={right}。
#
# 【公開測試資料 1】
# 輸入 / 設定：h = [9, 7, 0, 5, 2]
# 預期輸出：左鄰=7, 右鄰=5
#
# 【公開測試資料 2】
# 輸入 / 設定：h = [1, 0, 3]
# 預期輸出：左鄰=1, 右鄰=3
# ==============================================================================
# 請在下方撰寫你的程式碼並執行測試：

h = [9, 7, 0, 5, 2]  # 可自行替換為測試資料 2 進行驗證



In [ ]:
# ==============================================================================
# 14.9.2 [5] Code 挑戰題
# 任務說明：請撰寫一段代碼，統計給定的圍籬串列 h 中，
#           一共有多少個損壞的木樁（即 0 的總個數），
#           並將所有損壞木樁的索引位置依序存入一個清單 broken_indices 中印出！
# （本題為自由挑戰題，無公開測資，請依題目情境自行思考並撰寫完整程式碼）
# ==============================================================================
# 請在下方撰寫你的程式碼：



### 14.9.3 邊界特例分析：最左端第 0 格與最右端最後一格

#### 🌊 邊界的孤獨：當鄰居只有單邊存在
在官方範例 2 中，我們迎來了一組非常具備啟發性的測資：
`0 5 3 0 6 4 0 1 0`
請仔細觀察最前頭與最後頭：
- **第 0 格的高度是 `0`**；
- **最後一格（第 8 格）的高度也是 `0`**！

這兩個位置就是典型的**邊界端點（Endpoints）**。
題目對於這兩處給出了非常清晰且貼心的規則定義：
1. **最左端木樁（$i = 0$）**：
   它身處整排圍籬的最起點，左邊是無邊無際的荒野，根本沒有木樁。因此，它**唯一能參考的就只有右邊的鄰居（索引 1）**！
   所以修復高度直接就是右鄰居的高度：
   $$\text{Cost} = h[1]$$
2. **最右端木樁（$i = n - 1$）**：
   它身處整排圍籬的終點，右邊沒有任何木樁。因此，它**唯一能參考的就只有左邊的鄰居（索引 $n - 2$）**！
   所以修復高度直接就是左鄰居的高度：
   $$\text{Cost} = h[n - 2]$$

#### 🛡️ 傳統條件式防守心法（Three-way Branching）
在面對這種頭尾與中間規則不同的情境時，最穩健的做法就是使用 `if-elif-else` 進行三向互斥分流：
```python
if i == 0:
    cost = h[1]               # 最左端：只看右鄰居
elif i == n - 1:
    cost = h[n - 2]           # 最右端：只看左鄰居
else:
    cost = min(h[i-1], h[i+1]) # 中間格：左右鄰居取較小者
```
這種三段式分支邏輯層次分明、毫不模糊，完全杜絕了索引越界的致命崩潰！

In [ ]:
# ==============================================================================
# 14.9.3 [2] Code 範例區：頭端與尾端損壞特例的修復判定
# ==============================================================================

# 測試一：最左端損壞（i = 0）
h_left_broken = [0, 5, 8]
n1 = len(h_left_broken)
if h_left_broken[0] == 0:
    cost_left = h_left_broken[1]  # 參考右鄰居
    print(f"最左端損壞修復：參考右鄰居 h[1]={cost_left}")

# 測試二：最右端損壞（i = n - 1）
h_right_broken = [7, 6, 0]
n2 = len(h_right_broken)
if h_right_broken[n2 - 1] == 0:
    cost_right = h_right_broken[n2 - 2]  # 參考左鄰居
    print(f"最右端損壞修復：參考左鄰居 h[n-2]={cost_right}")

print("-" * 50)


In [ ]:
# ==============================================================================
# 14.9.3 [3] Code 填空題
# 任務說明：請補齊下方代碼中的挖空處 ___，完成針對邊界損壞位置的單獨費用計算函式。
# ==============================================================================

def get_boundary_cost(h, i, n):
    # 提示：若為最左端 i == 0，回傳右鄰居 h[1]
    if i == 0:
        return h[___]
    # 提示：若為最右端 i == n - 1，回傳左鄰居 h[n - 2]
    elif i == n - 1:
        return h[___]
    return 0

test_h = [0, 9, 3, 0]
print(f"最左端修復費：{get_boundary_cost(test_h, 0, 4)}")
print(f"最右端修復費：{get_boundary_cost(test_h, 3, 4)}")
# 預期輸出：最左端修復費：9，最右端修復費：3


In [ ]:
# ==============================================================================
# 14.9.3 [4] Code 練習題
# 任務說明：給定一個圍籬串列 h（長度為 2），其中恰有一根木樁損壞（為 0）。
#           請判斷是左端損壞還是右端損壞，並印出修復花費。
#
# 【公開測試資料 1】
# 輸入 / 設定：h = [0, 8]
# 預期輸出：8
# （推導：最左端損壞，修復花費等於右鄰居 8）
#
# 【公開測試資料 2】
# 輸入 / 設定：h = [5, 0]
# 預期輸出：5
# （推導：最右端損壞，修復花費等於左鄰居 5）
# ==============================================================================
# 請在下方撰寫你的程式碼並執行測試：

h = [0, 8]  # 可自行替換為測試資料 2 進行驗證



In [ ]:
# ==============================================================================
# 14.9.3 [5] Code 挑戰題
# 任務說明：給定一個圍籬串列 h = [0, 6, 4, 0]，頭尾兩端剛好「同時都損壞了」！
#           請撰寫程式碼，分別計算修復最左端與最右端的費用，並印出兩者的花費總和！
# （本題為自由挑戰題，無公開測資，請依題目情境自行思考並撰寫完整程式碼）
# ==============================================================================
# 請在下方撰寫你的程式碼：



### 14.9.4 中間損壞格取相鄰兩側較小值：`min(left, right)`

#### 🔗 統合三向分支的完整修復決策樹
現在，我們已經掌握了最左端、最右端以及中間一般格的所有計算規則。
每當我們在圍籬序列中巡查到一根損壞的木樁（`h[i] == 0`）時，我們可以建立一棵完整的修復決策樹：
```
                   h[i] == 0 發現損壞木樁
                            │
             ┌──────────────┼──────────────┐
             ▼              ▼              ▼
         i == 0         i == n - 1       其他中間位置
       （最左端）       （最右端）      （一般雙鄰居）
             │              │              │
        cost = h[1]    cost = h[n-2]   cost = min(h[i-1], h[i+1])
```

#### 💡 演算法實作細節
將決策樹轉化為程式碼結構：
```python
if h[i] == 0:
    if i == 0:
        cost = h[1]
    elif i == n - 1:
        cost = h[n - 2]
    else:
        cost = min(h[i - 1], h[i + 1])
```
這套判斷架構具有極佳的適應力：
- 如果圍籬只有 2 根木樁（$n = 2$），它只會走 `i == 0` 或 `i == n - 1`，絕不會誤入 `else`；
- 如果圍籬有數十根甚至上百根木樁，中間的所有位置都能安全調用 `min(h[i - 1], h[i + 1])`。
邏輯堅不可摧，完美應對所有測試邊界！

In [ ]:
# ==============================================================================
# 14.9.4 [2] Code 範例區：三向分支修補邏輯演算演示
# ==============================================================================

h = [0, 5, 3, 0, 6, 4, 0, 1, 0]  # 官方範例 2
n = len(h)

print(f"圍籬陣列：{h}（長度 n = {n}）")
print("=" * 55)

for i in range(n):
    if h[i] == 0:
        if i == 0:
            cost = h[1]
            pos_type = "最左端"
        elif i == n - 1:
            cost = h[n - 2]
            pos_type = "最右端"
        else:
            cost = min(h[i - 1], h[i + 1])
            pos_type = "中間格"
            
        print(f"位置 i = {i:2d}（{pos_type}）：修補花費 = {cost:2d}")

print("=" * 55)


In [ ]:
# ==============================================================================
# 14.9.4 [3] Code 填空題
# 任務說明：請補齊下方代碼中的挖空處 ___，完成計算任一損壞位置修復費用的函式。
# ==============================================================================

def calculate_single_cost(h, i, n):
    if i == 0:
        return h[1]
    elif i == ___:           # 提示：最右端索引為 n - 1
        return h[n - 2]
    else:
        return min(h[___], h[___])  # 提示：左鄰與右鄰取 min

demo_h = [2, 0, 4]
ans_cost = calculate_single_cost(demo_h, 1, 3)
print(f"中間損壞修復費：{ans_cost}")
# 預期輸出：中間損壞修復費：2


In [ ]:
# ==============================================================================
# 14.9.4 [4] Code 練習題
# 任務說明：給定一個圍籬陣列 h 與其長度 n，陣列中恰好有一個 0。
#           請使用完整三向分支邏輯找出該 0，並印出修復花費。
#
# 【公開測試資料 1】
# 輸入 / 設定：n = 5, h = [10, 20, 0, 30, 40]
# 預期輸出：20
# （推導：中間損壞，min(20, 30) = 20）
#
# 【公開測試資料 2】
# 輸入 / 設定：n = 3, h = [0, 15, 25]
# 預期輸出：15
# （推導：最左端損壞，參考右鄰居 15）
# ==============================================================================
# 請在下方撰寫你的程式碼並執行測試：

n = 5
h = [10, 20, 0, 30, 40]  # 可自行替換為測試資料 2 進行驗證



In [ ]:
# ==============================================================================
# 14.9.4 [5] Code 挑戰題
# 任務說明：除了計算費用，農場主人還希望我們能直接「在原地把圍籬修好」（將 0 改為修補後的高度）！
#           給定 h = [2, 0, 4]，請撰寫程式碼修補該串列，
#           並印出修補後完整的全新圍籬序列（例如 [2, 2, 4]）！
# （本題為自由挑戰題，無公開測資，請依題目情境自行思考並撰寫完整程式碼）
# ==============================================================================
# 請在下方撰寫你的程式碼：



### 14.9.5 總花費累加器與完整 AC 程式碼實作

#### 🧩 考場完整解題積木拼裝
現在，我們把所有的邊界判定與計算積木整合為一道在 APCS 考場上可以秒拿 100 分的完整解答！

整個程式的運作流程可劃分為三個標準樂章：
1. **第一樂章：讀取輸入資料**：
   - 第 1 行：木樁數量 $n$，`n = int(input())`
   - 第 2 行：木樁高度串列，`h = list(map(int, input().split()))`

2. **第二樂章：累加器走訪與分支結算**：
   - 初始化總費用累加器：`total_cost = 0`
   - 巡視每一根木樁：`for i in range(n):`
   - 若發現斷裂木樁（`h[i] == 0`）：
     - 若在最左端（`i == 0`）：`total_cost += h[1]`
     - 若在最右端（`i == n - 1`）：`total_cost += h[n - 2]`
     - 若在中間：`total_cost += min(h[i - 1], h[i + 1])`

3. **第三樂章：輸出最終總費用**：
   - `print(total_cost)`

#### 💡 線性時間與空間的優越性
這段程式碼僅需約 12 行，時間複雜度為嚴格的 $O(n)$，在 $n \le 100$ 的規模下只需走訪 100 次，耗時不到 0.0001 秒；空間複雜度為 $O(n)$，記憶體消耗極低，完美符合 APCS 官方評測的所有規格！

In [ ]:
# ==============================================================================
# 14.9.5 [2] Code 範例區：完整 AC 程式碼實作（以範例 2 模擬輸入）
# ==============================================================================

# 模擬輸入資料
n = 9
h = [0, 5, 3, 0, 6, 4, 0, 1, 0]

# 初始化總花費累加器
total_cost = 0

# 依序掃描所有木樁
for i in range(n):
    if h[i] == 0:
        if i == 0:
            total_cost += h[1]
        elif i == n - 1:
            total_cost += h[n - 2]
        else:
            total_cost += min(h[i - 1], h[i + 1])

# 輸出最終花費
print(f"APCS 評判系統預期輸出：{total_cost}")
# 驗證輸出：10，完全吻合！


In [ ]:
# ==============================================================================
# 14.9.5 [3] Code 填空題
# 任務說明：請補齊完整解題函式中的挖空處 ___，完成滿分解題核心！
# ==============================================================================

def solve_repair_fence(n, h):
    total = ___
    for i in range(n):
        if h[i] == 0:
            if i == 0:
                total += h[___]
            elif i == n - 1:
                total += h[___]
            else:
                total += min(h[___], h[___])
    return total

# 測試官方範例 1：預期輸出 2
print(f"範例 1 計算結果：{solve_repair_fence(3, [2, 0, 4])}")


In [ ]:
# ==============================================================================
# 14.9.5 [4] Code 練習題
# 任務說明：請撰寫完整的解題程式碼，計算並印出修復圍籬的總花費。
#
# 【公開測試資料 1】（官方範例 1）
# 輸入 / 設定：
# n = 3
# h = [2, 0, 4]
# 預期輸出：2
#
# 【公開測試資料 2】（官方範例 2）
# 輸入 / 設定：
# n = 9
# h = [0, 5, 3, 0, 6, 4, 0, 1, 0]
# 預期輸出：10
# ==============================================================================
# 請在下方撰寫你的程式碼並執行測試：

n = 3
h = [2, 0, 4]  # 可自行替換為測試資料 2 進行驗證



In [ ]:
# ==============================================================================
# 14.9.5 [5] Code 挑戰題
# 任務說明：如果木材行推出促銷方案：「如果總修復花費超過 20 元，全單享有 9 折優惠（整數除法 // 取整）」。
#           請擴充解題函式，在計算完總費用後套用此折扣，並印出最終實付金額！
# （本題為自由挑戰題，無公開測資，請依題目情境自行思考並撰寫完整程式碼）
# ==============================================================================
# 請在下方撰寫你的程式碼：



### 14.9.6 加框技巧延伸：前後各補虛擬圍籬免邊界特判

#### 🎩 競賽選手的黑魔法：哨兵加框法（Sentinel / Padding）
在先前的章節中，我們使用了 `if-elif-else` 來嚴密防守最左端與最右端的邊界。
然而，在競技程式設計（Competitive Programming）中，每多寫一個 `if` 條件分支，就多了一分打錯變數名稱的風險。
那麼，有沒有一種方法，**能讓最左端與最右端不需要寫任何特殊 `if` 判斷，直接套用通用的 `min(left, right)` 公式呢？**

答案就是無比優雅的——**「哨兵加框技巧」**！

#### 💡 哨兵數值的神奇魔力
讓我們思考：
- 當最左端 $i = 0$ 損壞時，我們希望它的費用是右鄰居 $h[1]$；
- 如果我們在它的左邊放置一根**「超級高的虛擬木樁」**（例如高度設為 `1000` 或 `float('inf')`）：
  $$\min(1000, h[1]) = h[1]$$
  因為 1000 比題目中任何真實木樁高度（最大 100）都要高，所以在取 `min` 時，這根虛擬木樁永遠不會勝出，**答案自然永遠是真實的右鄰居 $h[1]$！**
- 同理，在整排圍籬的最右端也補上一根 `1000` 的虛擬木樁，最右端的取值也會自然變成 $\min(h[n-2], 1000) = h[n-2]$！

在 Python 中，實現加框只需一行極簡語法：
```python
# 在頭尾各加上一個 1000 的虛擬哨兵
padded_h = [1000] + h + [1000]

total_cost = 0
# 原本的 h[0] ~ h[n-1] 現在對應 padded_h[1] ~ padded_h[n]
for i in range(1, n + 1):
    if padded_h[i] == 0:
        total_cost += min(padded_h[i - 1], padded_h[i + 1])
```
看！原本繁瑣的三向分支瞬間消失，所有位置全部統一為單一的 `min()` 運算！這項技巧在後續二維網格走訪（如地圖邊界保護）中更是不可或缺的神器！

In [ ]:
# ==============================================================================
# 14.9.6 [2] Code 範例區：哨兵加框法（Padding）極簡實作演示
# ==============================================================================

h = [0, 5, 3, 0, 6, 4, 0, 1, 0]  # 官方範例 2
n = len(h)

# 1. 前後各加上一根高度為 1000 的虛擬木樁
padded_h = [1000] + h + [1000]
print(f"原始圍籬：{h}")
print(f"加框後圍籬：{padded_h}")
print("-" * 55)

# 2. 一行迴圈無分支通殺所有位置
ans = 0
for i in range(1, n + 1):
    if padded_h[i] == 0:
        cost = min(padded_h[i - 1], padded_h[i + 1])
        ans += cost
        print(f"修補原始索引 {i - 1}：min({padded_h[i - 1]}, {padded_h[i + 1]}) = {cost}")

print("-" * 55)
print(f"加框法計算總花費：{ans}（完全免除 if-elif-else 特判！）")


In [ ]:
# ==============================================================================
# 14.9.6 [3] Code 填空題
# 任務說明：請補齊下方代碼中的挖空處 ___，完成極簡 5 行加框版解題函式！
# ==============================================================================

def solve_with_padding(n, h):
    # 提示：在前後各拼裝一個包含大數值 [999] 的串列
    ph = [___] + h + [___]
    cost = 0
    # 提示：走訪原陣列對應的範圍 1 到 n（包含 n）
    for i in range(1, ___):
        if ph[i] == 0:
            cost += min(ph[___], ph[___])
    return cost

print(f"加框法測試範例 1：{solve_with_padding(3, [2, 0, 4])}")
# 預期輸出：加框法測試範例 1：2


In [ ]:
# ==============================================================================
# 14.9.6 [4] Code 練習題
# 任務說明：請使用「加框技巧」撰寫程式碼，計算給定圍籬 h 的修復總花費。
#
# 【公開測試資料 1】
# 輸入 / 設定：
# n = 4
# h = [0, 8, 0, 6]
# 預期輸出：14
# （推導：索引 0 修復得 8，索引 2 修復得 min(8, 6) = 6，總計 8 + 6 = 14）
#
# 【公開測試資料 2】
# 輸入 / 設定：
# n = 2
# h = [0, 99]
# 預期輸出：99
# （推導：最左端修復得 99）
# ==============================================================================
# 請在下方撰寫你的程式碼並執行測試：

n = 4
h = [0, 8, 0, 6]  # 可自行替換為測試資料 2 進行驗證



In [ ]:
# ==============================================================================
# 14.9.6 [5] Code 挑戰題
# 任務說明：如果我們想挑戰極致的 Python 一行流（One-liner）！
#           利用加框串列 ph 與列表推導式，你能否用一行 sum(...) 算出總花費？
#           例如：sum(min(ph[i-1], ph[i+1]) for i in range(1, n+1) if ph[i] == 0)
#           請撰寫並測試範例 2，見證 Python 語法的強大表達力！
# （本題為自由挑戰題，無公開測資，請依題目情境自行思考並撰寫完整程式碼）
# ==============================================================================
# 請在下方撰寫你的程式碼：



## 🏆 恭喜通關！第十四章全數大滿貫、今日能力盤點與榮耀通關徽章

🎉 **狂賀！你已經成功征服了第十四章所有 9 道 APCS 實作初級真題！**
從 14.1 三角形辨別、14.2 秘密差、14.3 邏輯運算子、14.4 籃球比賽、14.5 購物車、14.6 人力分配、14.7 購買力、14.8 七言對聯，到今天的 **14.9 修補圍籬**，你已經徹底掌握了 APCS 實作初級題（2~3級分水準）的所有解題套路與考場避坑心法！

---

### 🌟 今日解鎖核心能力盤點
1. **相鄰元素探測模式**：熟練掌握一維陣列左右鄰居（`h[i-1]`, `h[i+1]`）的狀態提取技巧。
2. **極限邊界防禦心法**：深入理解最左端（`i == 0`）與最右端（`i == n - 1`）的邊界特性，徹底杜絕 `IndexError` 與負數索引倒轉。
3. **三向互斥分支架構**：以清晰的 `if-elif-else` 決策樹，實作高可讀性、零死角的滿分邏輯。
4. **黑魔法哨兵加框技法**：解鎖前後各補大數（`[1000] + h + [1000]`）的競賽秘技，免除所有邊界特判，程式碼極限精簡。
5. **第十四章大滿貫通關**：累積實戰經驗，具備從審題、抽象建模、避坑到雙平台 AC 的全方位實戰能力！

---

### 🎖️ 獲得榮耀通關徽章
```
╔═════════════════════════════════════════════════════════════════════════╗
║             🏆 APCS 實作初級真題（第十四章）全破大滿貫 🏆             ║
║                                                                         ║
║   【圍籬修繕神工匠】 × 【哨兵加框大師】 × 【APCS 初級真題大宗師】       ║
║                                                                         ║
║   榮譽認證：完破第 14 章全 9 道實作真題，扎下無可撼動的程式實作底氣！   ║
╚═════════════════════════════════════════════════════════════════════════╝
```

---

### 🚀 下一個壯闊征途預告
在接下來的 **第十五章 APCS 實作真題特訓（中級題，講義第 15 章共 10 題）** 中，我們將跨入 3~4 級分的進階殿堂！
首道登場的大題是 **15.1 b266. 矩陣轉換（APCS 2016-03 舊版實作第 2 題）**！我們將學習如何運用二維陣列、逆向操作序列倒序執行、90 度旋轉與垂直翻轉矩陣變換，敬請準備迎接更高層次的演算法挑戰！

## 💻 【附錄：雙平台滿分通關解答庫】考 APCS vs 刷 ZeroJudge 對照

許多同學在準備 APCS 時，常會困惑於「正式考場環境」與「線上刷題系統（如 ZeroJudge）」之間的程式寫法差異。
為了讓大家在兩種環境都能游刃有餘、輕鬆拿滿分，本筆記本特別提供兩種版本的標準通關原始碼：

| 評判平台 | 輸入機制特點 | 推薦程式寫法風格 | 核心優勢 |
| :--- | :--- | :--- | :--- |
| **🥇 APCS 正式考場** | 第一行輸入 $n$，第二行輸入 $n$ 個高度，保證單筆測資 | **極簡加框版 / 三向分支版**（直接呼叫 `input()`，不寫外層多餘迴圈） | 打字量極低（1 分鐘寫完）、結構清爽、考場得分神速 |
| **🥈 ZeroJudge 線上評判** | 測資伺服器常以**批次檔案串流（EOF）**連續灌入多筆測資 | **sys.stdin 批次解析版**（使用 `sys.stdin.read().split()`） | 徹底避免 `EOFError`，完全通殺線上 OJ 所有多測資模式 |

接下來的兩個儲存格分別提供了對應的完整程式碼，供初學同學在不同場合直接參考與取用！

### 📝 版本一：APCS 官方實作考場專用版（極簡 12 行，附逐行詳細註解）

> 📌 **考場使用指引**：  
> 本版本專為 **APCS 官方正式考場** 量身打造！  
> 正式考試時題目保證單筆測資輸入（第 1 行為 $n$，第 2 行為 $n$ 個整數）。
> 程式碼採用清晰的三向分支架構，**僅需約 12 行核心代碼**即可在 1 分鐘內快速寫完並獲得 100 分 AC！  
>
> 💡 **在 Colab 中互動測試**：  
> 點擊下方儲存格播放鍵，並在跳出的輸入框中依序貼上測試資料（按 Enter 逐行輸入），即可即時檢視結果。

In [ ]:
# ==============================================================================
# 📝 【版本一：APCS 官方實作考場專用版】
# 適用情境：APCS 正式考試環境（題目保證第 1 行為 n，第 2 行為 n 個木樁高度）
# ==============================================================================

# 1. 讀取木樁總數量 n
n = int(input())

# 2. 讀取木樁高度串列
h = list(map(int, input().split()))

# 3. 初始化總修補費用
total_cost = 0

# 4. 循序掃描每一根木樁
for i in range(n):
    # 若遇到損壞木樁（高度為 0）
    if h[i] == 0:
        if i == 0:
            # 最左端：參考右邊鄰居
            total_cost += h[1]
        elif i == n - 1:
            # 最右端：參考左邊鄰居
            total_cost += h[n - 2]
        else:
            # 中間位置：取左右鄰居中較小的高度
            total_cost += min(h[i - 1], h[i + 1])

# 5. 輸出修復總費用
print(total_cost)


### 🌐 版本二：ZeroJudge 線上評判萬用 AC 版（加框法 + 多測資 EOF 處理 + 自動化測試）

> 📌 **線上刷題指引**：  
> 在 ZeroJudge 等線上 OJ 系統中，伺服器通常會以重導向檔案連續灌入多筆測資。
> 本版本採用**「哨兵加框技巧」**封裝解題函式，並使用 `sys.stdin.read().split()` 一次性讀取所有數值 token，程式碼精簡至極且完全杜絕 `EOFError`！  
>
> 📋 **複製提交專區**：  
> 下方儲存格中特別劃分了【ZeroJudge 複製提交專區】，同學們只要複製該段程式碼，貼到 ZeroJudge g595 即可直接收穫 100% 滿分 AC！  
>
> 🧪 **內建自動測試驗證**：  
> 直接點擊執行下方儲存格，將自動驅動內建測試案例（包含官方範例 1、範例 2、全無損壞範例 3 與端點極小範例 4），無需手動打字即可看見綠燈通過報告！

In [ ]:
# ==============================================================================
# 🌐 【版本二：ZeroJudge 線上評判萬用 AC 版 + 本地自動化測試檢驗】
# ==============================================================================

# ------------------------------------------------------------------------------
# 📋 【ZeroJudge 複製提交專區】（可直接複製此函式與下方迴圈貼至 ZeroJudge g595）
# ------------------------------------------------------------------------------
def solve_fence(n, h):
    """
    核心解題演算法（加框法）：前後補 1000 虛擬木樁，避免邊界特判。
    時間複雜度：O(n)，空間複雜度：O(n)
    """
    ph = [1000] + h + [1000]
    total = 0
    for i in range(1, n + 1):
        if ph[i] == 0:
            total += min(ph[i - 1], ph[i + 1])
    return total

# 在 ZeroJudge 提交時使用的標準多測資 I/O 驅動骨架：
"""
import sys
tokens = sys.stdin.read().split()
idx = 0
while idx < len(tokens):
    n = int(tokens[idx])
    idx += 1
    h = [int(x) for x in tokens[idx : idx + n]]
    idx += n
    print(solve_fence(n, h))
"""

# ------------------------------------------------------------------------------
# 🧪 【Colab 本地自動化測試檢驗展示】：
# 點擊本儲存格播放鍵，將自動以官方四大範例進行全功能驗證！
# ------------------------------------------------------------------------------
print("=" * 65)
print("🚀 正在執行 APCS g595 官方範例本地檢驗...")
print("=" * 65)

test_samples = [
    (
        3, [2, 0, 4], 2,
        "官方範例 1：單一中間損壞，預期輸出 2"
    ),
    (
        9, [0, 5, 3, 0, 6, 4, 0, 1, 0], 10,
        "官方範例 2：涵蓋頭端、尾端與多處中間損壞，預期輸出 10"
    ),
    (
        4, [5, 6, 7, 8], 0,
        "極端邊界測試：全無損壞，預期輸出 0"
    ),
    (
        2, [0, 8], 8,
        "極端規模邊界：長度為 2 且端點損壞，預期輸出 8"
    )
]

all_passed = True
for idx, (n_val, h_data, expected, desc) in enumerate(test_samples, 1):
    actual = solve_fence(n_val, h_data)
    status = "✅ PASS" if actual == expected else "❌ FAIL"
    if actual != expected:
        all_passed = False
    print(f"\n【測資 {idx}】（{desc}）")
    print(f"輸入：n = {n_val}, 圍籬 = {h_data}")
    print(f"預期輸出：{expected} | 實際輸出：{actual} --> {status}")

print("\n" + "=" * 65)
if all_passed:
    print("🎉 本地 4 組測資檢驗全數通過！")
    print("學生可直接複製上方【ZeroJudge 複製提交專區】代碼至 ZeroJudge g595 取得 100% AC！")
else:
    print("⚠️ 有部分測資未通過，請檢查運算邏輯！")
print("=" * 65)
